# DeepSeek-V3 MoE Kernel — Complete Step-by-Step Walkthrough

Walks through **every operation** in `kernel_ref.py` with tiny concrete tensors.

| | Example | Production |
|---|---|---|
| T (tokens) | 4 | 4096 |
| H (hidden) | 4 | 7168 |
| I (intermediate) | 2 | 2048 |
| BLOCK (FP8 quant block) | 2 | 128 |
| E_global (total experts) | 8 | 256 |
| E_local (local experts) | 2 | 32 |
| N_GROUP | 2 | 8 |
| TOPK_GROUP | 1 | 4 |
| TOP_K | 2 | 8 |

**Phases:**
1. FP8 Block-Scale Dequantization (hidden states, W13, W2)
2. No-Aux Routing (sigmoid → group topk → global topk → weight normalization)
3. Per-Expert Compute Loop (token select → GEMM1 → SwiGLU → GEMM2 → weighted accum)
4. Cast to BF16

---

In [1]:
import torch
torch.manual_seed(42)

# ── Tiny dimensions ──
T = 4            # tokens
H = 4            # hidden size
I = 2            # intermediate size
BLOCK = 2        # FP8 quantization block size
E_global = 8     # total experts
E_local = 2      # local experts on this rank
N_GROUP = 2      # routing groups
TOPK_GROUP = 1   # top groups to keep
TOP_K = 2        # experts per token

group_size = E_global // N_GROUP  # 4 experts per group
num_h_blocks = H // BLOCK         # 2 scale blocks for hidden dim
num_2I_blocks = (2 * I) // BLOCK  # 2 scale blocks for gemm1 output dim
num_I_blocks = I // BLOCK         # 1 scale block for intermediate dim

local_expert_offset = 2  # our local experts are global experts 2 and 3
routed_scaling_factor = 1.0

print(f"Dimensions: T={T}, H={H}, I={I}, BLOCK={BLOCK}")
print(f"Experts: E_global={E_global}, E_local={E_local}, local_offset={local_expert_offset}")
print(f"  → local experts are global experts [{local_expert_offset}, {local_expert_offset + E_local})")
print(f"Routing: N_GROUP={N_GROUP}, group_size={group_size}, TOPK_GROUP={TOPK_GROUP}, TOP_K={TOP_K}")
print(f"Scale blocks: hidden={num_h_blocks}, gemm1_out={num_2I_blocks}, intermediate={num_I_blocks}")

Dimensions: T=4, H=4, I=2, BLOCK=2
Experts: E_global=8, E_local=2, local_offset=2
  → local experts are global experts [2, 4)
Routing: N_GROUP=2, group_size=4, TOPK_GROUP=1, TOP_K=2
Scale blocks: hidden=2, gemm1_out=2, intermediate=1


---
# INPUT TENSORS

These are what the kernel receives. In production they come from the model.

In [2]:
# ── Build all input tensors with small readable values ──

# Routing logits: [T, E_global] — raw logits before sigmoid
routing_logits = torch.tensor([
    [ 1.0, -1.0,  2.0,  3.0,  -2.0, 0.5, -0.5,  0.0],  # token 0
    [-1.0,  0.0,  1.5, -0.5,   2.5, 1.0,  0.0, -1.0],  # token 1
    [ 0.5,  1.0, -1.0,  0.0,   1.5, 2.0,  3.0, -0.5],  # token 2
    [ 2.0,  0.5,  1.0,  1.5,  -1.0, 0.0, -0.5,  0.5],  # token 3
], dtype=torch.float32)

# Routing bias: [E_global] — added to sigmoid scores
routing_bias = torch.tensor([0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=torch.float32)

# Hidden states: [T, H] — FP8 activations (we use float32 to simulate)
hidden_states = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [0.5, 1.5, 2.5, 3.5],
    [2.0, 1.0, 0.5, 1.5],
    [3.0, 2.0, 1.0, 0.5],
], dtype=torch.float32)

# Hidden states scale: [num_h_blocks, T] — note block-first!
hidden_states_scale = torch.tensor([
    [0.5, 0.4, 0.6, 0.3],   # block 0 scales for tokens 0-3
    [0.2, 0.3, 0.5, 0.4],   # block 1 scales for tokens 0-3
], dtype=torch.float32)

# GEMM1 weights: [E_local, 2*I, H] — gate+up projection (FP8)
gemm1_weights = torch.tensor([
    # Expert 0 (global expert 2): [2*I=4, H=4]
    [[ 1.0, 0.5, -0.5,  0.0],
     [ 0.0, 1.0,  0.5, -0.5],
     [ 0.5, 0.0,  1.0,  0.5],
     [-0.5, 0.5,  0.0,  1.0]],
    # Expert 1 (global expert 3): [2*I=4, H=4]
    [[ 0.5, 1.0,  0.0, -0.5],
     [-0.5, 0.0,  1.0,  0.5],
     [ 1.0, 0.5, -0.5,  0.0],
     [ 0.0, -0.5, 0.5,  1.0]],
], dtype=torch.float32)

# GEMM1 weight scales: [E_local, num_2I_blocks, num_h_blocks]
gemm1_weights_scale = torch.tensor([
    [[1.0, 0.8],    # expert 0: 2I_block0 × (h_block0, h_block1)
     [0.9, 1.1]],   # expert 0: 2I_block1 × (h_block0, h_block1)
    [[0.7, 1.0],    # expert 1
     [1.2, 0.6]],
], dtype=torch.float32)

# GEMM2 weights: [E_local, H, I] — down projection (FP8)
gemm2_weights = torch.tensor([
    # Expert 0: [H=4, I=2]
    [[ 0.5,  1.0],
     [ 1.0, -0.5],
     [-0.5,  0.5],
     [ 0.0,  1.0]],
    # Expert 1: [H=4, I=2]
    [[ 1.0,  0.0],
     [ 0.5,  1.0],
     [ 0.0, -0.5],
     [-0.5,  0.5]],
], dtype=torch.float32)

# GEMM2 weight scales: [E_local, num_h_blocks, num_I_blocks]
gemm2_weights_scale = torch.tensor([
    [[1.0],    # expert 0: h_block0 × I_block0
     [0.8]],   # expert 0: h_block1 × I_block0
    [[0.9],
     [1.1]],
], dtype=torch.float32)

print("INPUT TENSORS:")
for name, t in [("routing_logits", routing_logits), ("routing_bias", routing_bias),
                ("hidden_states", hidden_states), ("hidden_states_scale", hidden_states_scale),
                ("gemm1_weights", gemm1_weights), ("gemm1_weights_scale", gemm1_weights_scale),
                ("gemm2_weights", gemm2_weights), ("gemm2_weights_scale", gemm2_weights_scale)]:
    print(f"  {name:<25} {str(list(t.shape)):<18} {t.dtype}")

INPUT TENSORS:
  routing_logits            [4, 8]             torch.float32
  routing_bias              [8]                torch.float32
  hidden_states             [4, 4]             torch.float32
  hidden_states_scale       [2, 4]             torch.float32
  gemm1_weights             [2, 4, 4]          torch.float32
  gemm1_weights_scale       [2, 2, 2]          torch.float32
  gemm2_weights             [2, 4, 2]          torch.float32
  gemm2_weights_scale       [2, 2, 1]          torch.float32


---
# PHASE 1: FP8 Block-Scale Dequantization

Three tensors to dequantize: hidden states (A), GEMM1 weights (W13), GEMM2 weights (W2).

Each follows the same pattern: `dequantized = fp8_to_fp32(data) * expand_scale(scale)`

---
## Phase 1a: Dequant Hidden States

In [3]:
print("="*70)
print("PHASE 1a: Dequant hidden_states [T, H] using scale [num_h_blocks, T]")
print("="*70)

# Step 1: Cast FP8 → FP32
A_fp32 = hidden_states.to(torch.float32)
print(f"\nStep 1: A_fp32 = hidden_states.to(float32)")
print(f"  {list(hidden_states.shape)} FP8 → {list(A_fp32.shape)} FP32")
print(f"  A_fp32 =\n{A_fp32}")

# Step 2: Cast scale to FP32
A_scale = hidden_states_scale.to(torch.float32)
print(f"\nStep 2: A_scale = hidden_states_scale.to(float32)")
print(f"  {list(A_scale.shape)}  (rows=blocks, cols=tokens)")
print(f"  A_scale =\n{A_scale}")

# Step 3: Permute [num_blocks, T] → [T, num_blocks]
A_scale_TH = A_scale.permute(1, 0).contiguous()
print(f"\nStep 3: A_scale_TH = A_scale.permute(1,0).contiguous()")
print(f"  {list(A_scale.shape)} → {list(A_scale_TH.shape)}  (rows=tokens, cols=blocks)")
print(f"  A_scale_TH =\n{A_scale_TH}")
for t in range(T):
    print(f"    token {t}: scales = {A_scale_TH[t].tolist()} (block0, block1)")

# Step 4: Unsqueeze
A_scale_uns = A_scale_TH.unsqueeze(-1)
print(f"\nStep 4: .unsqueeze(-1)")
print(f"  {list(A_scale_TH.shape)} → {list(A_scale_uns.shape)}  (O(1) metadata, no copy)")

# Step 5: Repeat each scale BLOCK times
A_scale_rep = A_scale_uns.repeat(1, 1, BLOCK)
print(f"\nStep 5: .repeat(1, 1, BLOCK={BLOCK})")
print(f"  {list(A_scale_uns.shape)} → {list(A_scale_rep.shape)}")
print(f"  Each scale is copied {BLOCK} times along dim-2:")
for t in range(T):
    print(f"    token {t}: {A_scale_rep[t].tolist()}")

# Step 6: Reshape to [T, H]
A_scale_expanded = A_scale_rep.reshape(T, H)
print(f"\nStep 6: .reshape(T={T}, H={H})")
print(f"  {list(A_scale_rep.shape)} → {list(A_scale_expanded.shape)}  ({num_h_blocks}×{BLOCK}={H})")
print(f"  A_scale_expanded =\n{A_scale_expanded}")

# Step 7: Elementwise multiply
A = A_fp32 * A_scale_expanded
print(f"\nStep 7: A = A_fp32 * A_scale_expanded")
print(f"  A_fp32           =\n{A_fp32}")
print(f"  × A_scale_expanded =\n{A_scale_expanded}")
print(f"  = A              =\n{A}")
print(f"\n  e.g. A[0,0] = {A_fp32[0,0]:.1f} × {A_scale_expanded[0,0]:.1f} = {A[0,0]:.2f}")
print(f"       A[0,2] = {A_fp32[0,2]:.1f} × {A_scale_expanded[0,2]:.1f} = {A[0,2]:.2f}  (block 1, different scale)")

PHASE 1a: Dequant hidden_states [T, H] using scale [num_h_blocks, T]

Step 1: A_fp32 = hidden_states.to(float32)
  [4, 4] FP8 → [4, 4] FP32
  A_fp32 =
tensor([[1.0000, 2.0000, 3.0000, 4.0000],
        [0.5000, 1.5000, 2.5000, 3.5000],
        [2.0000, 1.0000, 0.5000, 1.5000],
        [3.0000, 2.0000, 1.0000, 0.5000]])

Step 2: A_scale = hidden_states_scale.to(float32)
  [2, 4]  (rows=blocks, cols=tokens)
  A_scale =
tensor([[0.5000, 0.4000, 0.6000, 0.3000],
        [0.2000, 0.3000, 0.5000, 0.4000]])

Step 3: A_scale_TH = A_scale.permute(1,0).contiguous()
  [2, 4] → [4, 2]  (rows=tokens, cols=blocks)
  A_scale_TH =
tensor([[0.5000, 0.2000],
        [0.4000, 0.3000],
        [0.6000, 0.5000],
        [0.3000, 0.4000]])
    token 0: scales = [0.5, 0.20000000298023224] (block0, block1)
    token 1: scales = [0.4000000059604645, 0.30000001192092896] (block0, block1)
    token 2: scales = [0.6000000238418579, 0.5] (block0, block1)
    token 3: scales = [0.30000001192092896, 0.400000005960464

---
## Phase 1b: Dequant GEMM1 Weights (W13)

Same pattern but 3D: `[E_local, 2*I, H]` with scale `[E_local, num_2I_blocks, num_h_blocks]`.

Scale is expanded along **both** dim-1 (rows) and dim-2 (cols) by BLOCK.

In [4]:
print("="*70)
print("PHASE 1b: Dequant gemm1_weights [E_local, 2*I, H]")
print("="*70)

W13_fp32 = gemm1_weights.to(torch.float32)
S13 = gemm1_weights_scale.to(torch.float32)
print(f"\nW13_fp32 shape: {list(W13_fp32.shape)}  [E_local={E_local}, 2I={2*I}, H={H}]")
print(f"S13 shape:      {list(S13.shape)}  [E_local={E_local}, 2I_blocks={num_2I_blocks}, H_blocks={num_h_blocks}]")
print(f"\nS13 (scale per block) =\n{S13}")

# Expand dim 1: repeat_interleave(BLOCK, dim=1)
S13_d1 = torch.repeat_interleave(S13, BLOCK, dim=1)
print(f"\nStep 1: repeat_interleave(BLOCK={BLOCK}, dim=1)")
print(f"  {list(S13.shape)} → {list(S13_d1.shape)}")
print(f"  Each row-block scale is repeated {BLOCK} times along rows:")
print(f"  S13_d1[expert=0] =\n{S13_d1[0]}")

# Expand dim 2: repeat_interleave(BLOCK, dim=2)
S13_d2 = torch.repeat_interleave(S13_d1, BLOCK, dim=2)
print(f"\nStep 2: repeat_interleave(BLOCK={BLOCK}, dim=2)")
print(f"  {list(S13_d1.shape)} → {list(S13_d2.shape)}  (matches W13_fp32)")
print(f"  S13_d2[expert=0] =\n{S13_d2[0]}")

# Multiply
W13 = W13_fp32 * S13_d2
print(f"\nStep 3: W13 = W13_fp32 * S13_expanded")
print(f"  W13_fp32[expert=0] =\n{W13_fp32[0]}")
print(f"  × S13_d2[expert=0] =\n{S13_d2[0]}")
print(f"  = W13[expert=0]    =\n{W13[0]}")
print(f"\n  e.g. W13[0,0,0] = {W13_fp32[0,0,0]:.1f} × {S13_d2[0,0,0]:.1f} = {W13[0,0,0]:.2f}")
print(f"       W13[0,0,2] = {W13_fp32[0,0,2]:.1f} × {S13_d2[0,0,2]:.1f} = {W13[0,0,2]:.2f}  (h_block=1)")
print(f"       W13[0,2,0] = {W13_fp32[0,2,0]:.1f} × {S13_d2[0,2,0]:.1f} = {W13[0,2,0]:.2f}  (2I_block=1)")

PHASE 1b: Dequant gemm1_weights [E_local, 2*I, H]

W13_fp32 shape: [2, 4, 4]  [E_local=2, 2I=4, H=4]
S13 shape:      [2, 2, 2]  [E_local=2, 2I_blocks=2, H_blocks=2]

S13 (scale per block) =
tensor([[[1.0000, 0.8000],
         [0.9000, 1.1000]],

        [[0.7000, 1.0000],
         [1.2000, 0.6000]]])

Step 1: repeat_interleave(BLOCK=2, dim=1)
  [2, 2, 2] → [2, 4, 2]
  Each row-block scale is repeated 2 times along rows:
  S13_d1[expert=0] =
tensor([[1.0000, 0.8000],
        [1.0000, 0.8000],
        [0.9000, 1.1000],
        [0.9000, 1.1000]])

Step 2: repeat_interleave(BLOCK=2, dim=2)
  [2, 4, 2] → [2, 4, 4]  (matches W13_fp32)
  S13_d2[expert=0] =
tensor([[1.0000, 1.0000, 0.8000, 0.8000],
        [1.0000, 1.0000, 0.8000, 0.8000],
        [0.9000, 0.9000, 1.1000, 1.1000],
        [0.9000, 0.9000, 1.1000, 1.1000]])

Step 3: W13 = W13_fp32 * S13_expanded
  W13_fp32[expert=0] =
tensor([[ 1.0000,  0.5000, -0.5000,  0.0000],
        [ 0.0000,  1.0000,  0.5000, -0.5000],
        [ 0.5000,  

---
## Phase 1c: Dequant GEMM2 Weights (W2)

Same pattern: `[E_local, H, I]` with scale `[E_local, num_h_blocks, num_I_blocks]`.

In [5]:
print("="*70)
print("PHASE 1c: Dequant gemm2_weights [E_local, H, I]")
print("="*70)

W2_fp32 = gemm2_weights.to(torch.float32)
S2 = gemm2_weights_scale.to(torch.float32)

S2_d1 = torch.repeat_interleave(S2, BLOCK, dim=1)
S2_d2 = torch.repeat_interleave(S2_d1, BLOCK, dim=2)
W2 = W2_fp32 * S2_d2

print(f"\nW2_fp32: {list(W2_fp32.shape)}, S2: {list(S2.shape)}")
print(f"S2 → expand dim1 → {list(S2_d1.shape)} → expand dim2 → {list(S2_d2.shape)}")
print(f"\nW2_fp32[expert=0] =\n{W2_fp32[0]}")
print(f"× S2_expanded[expert=0] =\n{S2_d2[0]}")
print(f"= W2[expert=0]          =\n{W2[0]}")

print(f"\n{'='*70}")
print(f"PHASE 1 COMPLETE — Dequantized tensors:")
print(f"  A:   {list(A.shape)} FP32   (hidden states)")
print(f"  W13: {list(W13.shape)} FP32   (gemm1 weights, per expert)")
print(f"  W2:  {list(W2.shape)} FP32   (gemm2 weights, per expert)")
print(f"{'='*70}")

PHASE 1c: Dequant gemm2_weights [E_local, H, I]

W2_fp32: [2, 4, 2], S2: [2, 2, 1]
S2 → expand dim1 → [2, 4, 1] → expand dim2 → [2, 4, 2]

W2_fp32[expert=0] =
tensor([[ 0.5000,  1.0000],
        [ 1.0000, -0.5000],
        [-0.5000,  0.5000],
        [ 0.0000,  1.0000]])
× S2_expanded[expert=0] =
tensor([[1.0000, 1.0000],
        [1.0000, 1.0000],
        [0.8000, 0.8000],
        [0.8000, 0.8000]])
= W2[expert=0]          =
tensor([[ 0.5000,  1.0000],
        [ 1.0000, -0.5000],
        [-0.4000,  0.4000],
        [ 0.0000,  0.8000]])

PHASE 1 COMPLETE — Dequantized tensors:
  A:   [4, 4] FP32   (hidden states)
  W13: [2, 4, 4] FP32   (gemm1 weights, per expert)
  W2:  [2, 4, 2] FP32   (gemm2 weights, per expert)


---
# PHASE 2: No-Aux Routing (DeepSeek-V3)

Determines which experts each token goes to, and with what weight.

**Flow:** logits → sigmoid → add bias → group scores → top groups → global top-k → normalize weights

---
## Step 2.1: Sigmoid

In [6]:
print("="*70)
print("STEP 2.1: Sigmoid")
print("="*70)

logits = routing_logits.to(torch.float32)
print(f"\nInput: routing_logits {list(logits.shape)} [T, E_global]")
print(f"{logits}")

s = 1.0 / (1.0 + torch.exp(-logits))
print(f"\nOperation: s = sigmoid(logits) = 1/(1+exp(-logits))")
print(f"Output: s {list(s.shape)} [T, E_global]")
print(f"{s}")
print(f"\n  e.g. token 0, expert 0: sigmoid({logits[0,0]:.1f}) = 1/(1+exp(-{logits[0,0]:.1f})) = {s[0,0]:.4f}")
print(f"       token 0, expert 1: sigmoid({logits[0,1]:.1f}) = 1/(1+exp({logits[0,1].abs():.1f}))  = {s[0,1]:.4f}")
print(f"\n  Sigmoid squashes all values to (0, 1). Higher logit → higher score.")

STEP 2.1: Sigmoid

Input: routing_logits [4, 8] [T, E_global]
tensor([[ 1.0000, -1.0000,  2.0000,  3.0000, -2.0000,  0.5000, -0.5000,  0.0000],
        [-1.0000,  0.0000,  1.5000, -0.5000,  2.5000,  1.0000,  0.0000, -1.0000],
        [ 0.5000,  1.0000, -1.0000,  0.0000,  1.5000,  2.0000,  3.0000, -0.5000],
        [ 2.0000,  0.5000,  1.0000,  1.5000, -1.0000,  0.0000, -0.5000,  0.5000]])

Operation: s = sigmoid(logits) = 1/(1+exp(-logits))
Output: s [4, 8] [T, E_global]
tensor([[0.7311, 0.2689, 0.8808, 0.9526, 0.1192, 0.6225, 0.3775, 0.5000],
        [0.2689, 0.5000, 0.8176, 0.3775, 0.9241, 0.7311, 0.5000, 0.2689],
        [0.6225, 0.7311, 0.2689, 0.5000, 0.8176, 0.8808, 0.9526, 0.3775],
        [0.8808, 0.6225, 0.7311, 0.8176, 0.2689, 0.5000, 0.3775, 0.6225]])

  e.g. token 0, expert 0: sigmoid(1.0) = 1/(1+exp(-1.0)) = 0.7311
       token 0, expert 1: sigmoid(-1.0) = 1/(1+exp(1.0))  = 0.2689

  Sigmoid squashes all values to (0, 1). Higher logit → higher score.


---
## Step 2.2: Add Bias

In [7]:
print("="*70)
print("STEP 2.2: Add Bias")
print("="*70)

bias = routing_bias.to(torch.float32).reshape(-1)
print(f"\nInput: s {list(s.shape)}, bias {list(bias.shape)}")
print(f"bias = {bias.tolist()}")
print(f"  (only expert 2 has nonzero bias = {bias[2]:.1f})")

s_with_bias = s + bias
print(f"\nOperation: s_with_bias = s + bias  (broadcast [T,E] + [E])")
print(f"Output: s_with_bias {list(s_with_bias.shape)}")
print(f"{s_with_bias}")
print(f"\n  Bias boosts/penalizes certain experts in the selection step.")
print(f"  Expert 2 got +0.1 boost for all tokens.")
print(f"  e.g. token 0, expert 2: {s[0,2]:.4f} + {bias[2]:.1f} = {s_with_bias[0,2]:.4f}")

STEP 2.2: Add Bias

Input: s [4, 8], bias [8]
bias = [0.0, 0.0, 0.10000000149011612, 0.0, 0.0, 0.0, 0.0, 0.0]
  (only expert 2 has nonzero bias = 0.1)

Operation: s_with_bias = s + bias  (broadcast [T,E] + [E])
Output: s_with_bias [4, 8]
tensor([[0.7311, 0.2689, 0.9808, 0.9526, 0.1192, 0.6225, 0.3775, 0.5000],
        [0.2689, 0.5000, 0.9176, 0.3775, 0.9241, 0.7311, 0.5000, 0.2689],
        [0.6225, 0.7311, 0.3689, 0.5000, 0.8176, 0.8808, 0.9526, 0.3775],
        [0.8808, 0.6225, 0.8311, 0.8176, 0.2689, 0.5000, 0.3775, 0.6225]])

  Bias boosts/penalizes certain experts in the selection step.
  Expert 2 got +0.1 boost for all tokens.
  e.g. token 0, expert 2: 0.8808 + 0.1 = 0.9808


---
## Step 2.3: Group Scoring

Experts are divided into N_GROUP=2 groups of group_size=4. For each group, take the top-2 scores and sum them → group score.

In [8]:
print("="*70)
print("STEP 2.3: Group Scoring")
print("="*70)

# View as groups
s_grouped = s_with_bias.view(T, N_GROUP, group_size)
print(f"\nOperation: s_with_bias.view(T={T}, N_GROUP={N_GROUP}, group_size={group_size})")
print(f"  {list(s_with_bias.shape)} → {list(s_grouped.shape)}")
print(f"\n  Group 0 = experts [0,1,2,3],  Group 1 = experts [4,5,6,7]")
for t in range(T):
    print(f"  token {t}: group0={[f'{v:.3f}' for v in s_grouped[t,0].tolist()]}  "
          f"group1={[f'{v:.3f}' for v in s_grouped[t,1].tolist()]}")

# Top-2 per group
top2_vals, top2_idx = torch.topk(s_grouped, k=2, dim=2, largest=True, sorted=False)
print(f"\nOperation: topk(k=2, dim=2) — top 2 scores within each group")
print(f"Output: top2_vals {list(top2_vals.shape)}")
for t in range(T):
    print(f"  token {t}: group0 top2={[f'{v:.3f}' for v in top2_vals[t,0].tolist()]}  "
          f"group1 top2={[f'{v:.3f}' for v in top2_vals[t,1].tolist()]}")

# Group scores = sum of top-2
group_scores = top2_vals.sum(dim=2)
print(f"\nOperation: group_scores = top2_vals.sum(dim=2)")
print(f"Output: group_scores {list(group_scores.shape)}")
for t in range(T):
    print(f"  token {t}: group0={group_scores[t,0]:.4f}  group1={group_scores[t,1]:.4f}")

STEP 2.3: Group Scoring

Operation: s_with_bias.view(T=4, N_GROUP=2, group_size=4)
  [4, 8] → [4, 2, 4]

  Group 0 = experts [0,1,2,3],  Group 1 = experts [4,5,6,7]
  token 0: group0=['0.731', '0.269', '0.981', '0.953']  group1=['0.119', '0.622', '0.378', '0.500']
  token 1: group0=['0.269', '0.500', '0.918', '0.378']  group1=['0.924', '0.731', '0.500', '0.269']
  token 2: group0=['0.622', '0.731', '0.369', '0.500']  group1=['0.818', '0.881', '0.953', '0.378']
  token 3: group0=['0.881', '0.622', '0.831', '0.818']  group1=['0.269', '0.500', '0.378', '0.622']

Operation: topk(k=2, dim=2) — top 2 scores within each group
Output: top2_vals [4, 2, 2]
  token 0: group0 top2=['0.981', '0.953']  group1 top2=['0.622', '0.500']
  token 1: group0 top2=['0.918', '0.500']  group1 top2=['0.924', '0.731']
  token 2: group0 top2=['0.731', '0.622']  group1 top2=['0.953', '0.881']
  token 3: group0 top2=['0.881', '0.831']  group1 top2=['0.622', '0.500']

Operation: group_scores = top2_vals.sum(dim=2)
O

---
## Step 2.4: Select Top Groups

In [9]:
print("="*70)
print("STEP 2.4: Select Top Groups")
print("="*70)

_, group_idx = torch.topk(group_scores, k=TOPK_GROUP, dim=1, largest=True, sorted=False)
print(f"\nOperation: topk(k=TOPK_GROUP={TOPK_GROUP}, dim=1) on group_scores")
print(f"Output: group_idx {list(group_idx.shape)} — which group(s) each token keeps")
for t in range(T):
    g = group_idx[t].tolist()
    print(f"  token {t}: keeps group {g}  "
          f"(score={group_scores[t, g[0]]:.4f} vs dropped={group_scores[t, 1-g[0]]:.4f})")

# Build group mask
group_mask = torch.zeros_like(group_scores)
group_mask.scatter_(1, group_idx, 1.0)
print(f"\nOperation: scatter_(1, group_idx, 1.0) → group_mask {list(group_mask.shape)}")
print(f"{group_mask}")

# Expand to expert-level mask
score_mask = group_mask.unsqueeze(2).expand(T, N_GROUP, group_size).reshape(T, E_global)
print(f"\nOperation: unsqueeze(2).expand({T},{N_GROUP},{group_size}).reshape({T},{E_global})")
print(f"  {list(group_mask.shape)} → {list(score_mask.shape)}")
print(f"score_mask =\n{score_mask}")
print(f"\n  1 = expert is in a kept group, 0 = expert is masked out")

STEP 2.4: Select Top Groups

Operation: topk(k=TOPK_GROUP=1, dim=1) on group_scores
Output: group_idx [4, 1] — which group(s) each token keeps
  token 0: keeps group [0]  (score=1.9334 vs dropped=1.1225)
  token 1: keeps group [1]  (score=1.6552 vs dropped=1.4176)
  token 2: keeps group [1]  (score=1.8334 vs dropped=1.3535)
  token 3: keeps group [0]  (score=1.7119 vs dropped=1.1225)

Operation: scatter_(1, group_idx, 1.0) → group_mask [4, 2]
tensor([[1., 0.],
        [0., 1.],
        [0., 1.],
        [1., 0.]])

Operation: unsqueeze(2).expand(4,2,4).reshape(4,8)
  [4, 2] → [4, 8]
score_mask =
tensor([[1., 1., 1., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0., 0., 0., 0.]])

  1 = expert is in a kept group, 0 = expert is masked out


---
## Step 2.5: Global Top-K Selection

In [10]:
print("="*70)
print("STEP 2.5: Global Top-K Expert Selection")
print("="*70)

# Mask out experts from non-selected groups
neg_inf = torch.finfo(torch.float32).min
scores_pruned = s_with_bias.masked_fill(score_mask == 0, neg_inf)
print(f"\nOperation: s_with_bias.masked_fill(score_mask==0, -inf)")
print(f"  Experts in dropped groups get -inf so they can't be selected.")
print(f"scores_pruned (showing -inf as '____'):")
for t in range(T):
    vals = [f'{v:.3f}' if v > -1e30 else ' ____' for v in scores_pruned[t].tolist()]
    print(f"  token {t}: [{', '.join(vals)}]")

# Global top-k
_, topk_idx = torch.topk(scores_pruned, k=TOP_K, dim=1, largest=True, sorted=False)
print(f"\nOperation: topk(k=TOP_K={TOP_K}, dim=1) on scores_pruned")
print(f"Output: topk_idx {list(topk_idx.shape)} — which experts each token selects")
for t in range(T):
    experts = topk_idx[t].tolist()
    scores = [s_with_bias[t, e].item() for e in experts]
    print(f"  token {t}: experts {experts}  (biased scores: {[f'{v:.4f}' for v in scores]})")

STEP 2.5: Global Top-K Expert Selection

Operation: s_with_bias.masked_fill(score_mask==0, -inf)
  Experts in dropped groups get -inf so they can't be selected.
scores_pruned (showing -inf as '____'):
  token 0: [0.731, 0.269, 0.981, 0.953,  ____,  ____,  ____,  ____]
  token 1: [ ____,  ____,  ____,  ____, 0.924, 0.731, 0.500, 0.269]
  token 2: [ ____,  ____,  ____,  ____, 0.818, 0.881, 0.953, 0.378]
  token 3: [0.881, 0.622, 0.831, 0.818,  ____,  ____,  ____,  ____]

Operation: topk(k=TOP_K=2, dim=1) on scores_pruned
Output: topk_idx [4, 2] — which experts each token selects
  token 0: experts [2, 3]  (biased scores: ['0.9808', '0.9526'])
  token 1: experts [4, 5]  (biased scores: ['0.9241', '0.7311'])
  token 2: experts [6, 5]  (biased scores: ['0.9526', '0.8808'])
  token 3: experts [0, 2]  (biased scores: ['0.8808', '0.8311'])


---
## Step 2.6: Compute Routing Weights

**Key:** weights use `s` (without bias), not `s_with_bias`. Bias only affects *selection*, not *weighting*.

In [11]:
print("="*70)
print("STEP 2.6: Compute Routing Weights")
print("="*70)

# Build binary mask from topk_idx
M = torch.zeros_like(s)
M.scatter_(1, topk_idx, 1.0)
print(f"\nOperation: M = zeros({T},{E_global}); M.scatter_(1, topk_idx, 1.0)")
print(f"M (1 where expert is selected) =\n{M}")

# Weights = s (unbiased!) * mask
weights_raw = s * M
print(f"\nOperation: weights = s * M  (using s WITHOUT bias)")
print(f"weights_raw =\n{weights_raw}")

# Normalize
weights_sum = weights_raw.sum(dim=1, keepdim=True) + 1e-20
weights = (weights_raw / weights_sum) * routed_scaling_factor
print(f"\nOperation: weights = (weights / sum) * routed_scaling_factor={routed_scaling_factor}")
print(f"weights_sum per token = {weights_sum.squeeze().tolist()}")
print(f"\nweights (normalized) =\n{weights}")

print(f"\nPer-token summary:")
for t in range(T):
    experts = topk_idx[t].tolist()
    wts = [weights[t, e].item() for e in experts]
    print(f"  token {t}: experts {experts}, weights {[f'{w:.4f}' for w in wts]}, sum={sum(wts):.4f}")

print(f"\n{'='*70}")
print(f"PHASE 2 COMPLETE — Routing results:")
print(f"  topk_idx: {list(topk_idx.shape)} — selected expert IDs per token")
print(f"  weights:  {list(weights.shape)} — normalized routing weights")
print(f"{'='*70}")

STEP 2.6: Compute Routing Weights

Operation: M = zeros(4,8); M.scatter_(1, topk_idx, 1.0)
M (1 where expert is selected) =
tensor([[0., 0., 1., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1., 1., 0.],
        [1., 0., 1., 0., 0., 0., 0., 0.]])

Operation: weights = s * M  (using s WITHOUT bias)
weights_raw =
tensor([[0.0000, 0.0000, 0.8808, 0.9526, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.9241, 0.7311, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8808, 0.9526, 0.0000],
        [0.8808, 0.0000, 0.7311, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

Operation: weights = (weights / sum) * routed_scaling_factor=1.0
weights_sum per token = [1.8333711624145508, 1.655200481414795, 1.8333711624145508, 1.6118556261062622]

weights (normalized) =
tensor([[0.0000, 0.0000, 0.4804, 0.5196, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5583, 0.4417, 0.0000, 0.0000],
  

---
# PHASE 3: Per-Expert Compute Loop

For each of the E_local=2 local experts, find which tokens selected it, then:

**GEMM1** → **SwiGLU** → **GEMM2** → **Weighted Accumulate**

---
## Initialize output

In [12]:
output_fp32 = torch.zeros(T, H, dtype=torch.float32)
print(f"output = zeros({T}, {H}) FP32")
print(f"  Each expert's contribution will be added to this.")
print(f"\nLocal experts: global IDs {local_expert_offset} to {local_expert_offset + E_local - 1}")
print(f"\nWhich tokens selected which experts?")
for t in range(T):
    print(f"  token {t} selected experts {topk_idx[t].tolist()}")

output = zeros(4, 4) FP32
  Each expert's contribution will be added to this.

Local experts: global IDs 2 to 3

Which tokens selected which experts?
  token 0 selected experts [2, 3]
  token 1 selected experts [4, 5]
  token 2 selected experts [6, 5]
  token 3 selected experts [0, 2]


---
## Expert Loop — Iteration by Iteration

In [13]:
for le in range(E_local):
    ge = local_expert_offset + le  # global expert ID
    print(f"\n{'#'*70}")
    print(f"EXPERT le={le}, ge={ge} (global expert {ge})")
    print(f"{'#'*70}")

    # ── Step 3a: Token Selection ──
    print(f"\n--- Step 3a: Token Selection ---")
    sel_mask = (topk_idx == ge).any(dim=1)
    print(f"Operation: sel_mask = (topk_idx == {ge}).any(dim=1)")
    print(f"  topk_idx =\n{topk_idx}")
    print(f"  (topk_idx == {ge}) =\n{(topk_idx == ge)}")
    print(f"  .any(dim=1) = {sel_mask.tolist()}")

    if not sel_mask.any():
        print(f"  → No tokens selected expert {ge}. SKIP.")
        continue

    token_idx = torch.nonzero(sel_mask, as_tuple=False).squeeze(1)
    Tk = token_idx.numel()
    print(f"\n  token_idx = nonzero(sel_mask) = {token_idx.tolist()}")
    print(f"  → {Tk} token(s) selected expert {ge}")

    # ── Step 3b: Gather ──
    print(f"\n--- Step 3b: Gather inputs ---")
    A_e = A.index_select(0, token_idx)
    W13_e = W13[le]
    W2_e = W2[le]
    print(f"  A_e = A[token_idx] = A[{token_idx.tolist()}]")
    print(f"    A_e {list(A_e.shape)} =\n{A_e}")
    print(f"  W13_e = W13[{le}] {list(W13_e.shape)} =\n{W13_e}")
    print(f"  W2_e  = W2[{le}]  {list(W2_e.shape)} =\n{W2_e}")

    # ── Step 3c: GEMM1 ──
    print(f"\n--- Step 3c: GEMM1 ---")
    print(f"  Operation: G1 = A_e @ W13_e.T")
    print(f"    [{Tk}, {H}] @ [{H}, {2*I}] → [{Tk}, {2*I}]")
    G1 = A_e.matmul(W13_e.t())
    print(f"  G1 =\n{G1}")
    for i in range(Tk):
        print(f"    G1[{i}] = A_e[{i}]·W13_e.T = dot products of {A_e[i].tolist()} with each column")

    # ── Step 3d: SwiGLU ──
    print(f"\n--- Step 3d: SwiGLU (split → silu → multiply) ---")

    X1 = G1[:, :I]
    X2 = G1[:, I:]
    print(f"  Split G1 [{Tk},{2*I}] into:")
    print(f"    X1 = G1[:, :{I}] = {X1.tolist()}  (first {I} cols)")
    print(f"    X2 = G1[:, {I}:] = {X2.tolist()}  (last {I} cols)")

    silu_X2 = X2 / (1.0 + torch.exp(-X2))
    print(f"\n  SiLU(X2) = X2 / (1 + exp(-X2)):")
    for i in range(Tk):
        for j in range(I):
            x = X2[i,j].item()
            r = silu_X2[i,j].item()
            print(f"    silu({x:.4f}) = {x:.4f} / (1 + exp(-{x:.4f})) = {r:.4f}")
    print(f"  silu_X2 = {silu_X2.tolist()}")

    C = silu_X2 * X1
    print(f"\n  C = silu(X2) * X1  (elementwise)")
    for i in range(Tk):
        for j in range(I):
            print(f"    C[{i},{j}] = {silu_X2[i,j]:.4f} * {X1[i,j]:.4f} = {C[i,j]:.4f}")
    print(f"  C {list(C.shape)} = {C.tolist()}")

    # ── Step 3e: GEMM2 ──
    print(f"\n--- Step 3e: GEMM2 ---")
    print(f"  Operation: O = C @ W2_e.T")
    print(f"    [{Tk}, {I}] @ [{I}, {H}] → [{Tk}, {H}]")
    O = C.matmul(W2_e.t())
    print(f"  O =\n{O}")

    # ── Step 3f: Weighted Accumulation ──
    print(f"\n--- Step 3f: Weighted Accumulation ---")
    w_tok = weights.index_select(0, token_idx)[:, ge]
    print(f"  w_tok = weights[{token_idx.tolist()}, {ge}] = {w_tok.tolist()}")

    O_weighted = O * w_tok.unsqueeze(1)
    print(f"\n  O_weighted = O * w_tok.unsqueeze(1)")
    for i in range(Tk):
        tid = token_idx[i].item()
        print(f"    token {tid}: {O[i].tolist()} * {w_tok[i]:.4f} = {O_weighted[i].tolist()}")

    output_fp32.index_add_(0, token_idx, O_weighted)
    print(f"\n  output.index_add_(0, {token_idx.tolist()}, O_weighted)")
    print(f"  output (after expert {ge}) =\n{output_fp32}")


######################################################################
EXPERT le=0, ge=2 (global expert 2)
######################################################################

--- Step 3a: Token Selection ---
Operation: sel_mask = (topk_idx == 2).any(dim=1)
  topk_idx =
tensor([[2, 3],
        [4, 5],
        [6, 5],
        [0, 2]])
  (topk_idx == 2) =
tensor([[ True, False],
        [False, False],
        [False, False],
        [False,  True]])
  .any(dim=1) = [True, False, False, True]

  token_idx = nonzero(sel_mask) = [0, 3]
  → 2 token(s) selected expert 2

--- Step 3b: Gather inputs ---
  A_e = A[token_idx] = A[[0, 3]]
    A_e [2, 4] =
tensor([[0.5000, 1.0000, 0.6000, 0.8000],
        [0.9000, 0.6000, 0.4000, 0.2000]])
  W13_e = W13[0] [4, 4] =
tensor([[ 1.0000,  0.5000, -0.4000,  0.0000],
        [ 0.0000,  1.0000,  0.4000, -0.4000],
        [ 0.4500,  0.0000,  1.1000,  0.5500],
        [-0.4500,  0.4500,  0.0000,  1.1000]])
  W2_e  = W2[0]  [4, 2] =
tensor([[ 0.5000,  1.

---
# PHASE 4: Cast to BF16

In [14]:
print("="*70)
print("PHASE 4: Cast to BF16")
print("="*70)

output = output_fp32.to(torch.bfloat16)
print(f"\nOperation: output.to(torch.bfloat16)")
print(f"  {list(output_fp32.shape)} FP32 → {list(output.shape)} BF16")
print(f"\nFP32 output =\n{output_fp32}")
print(f"\nBF16 output =\n{output}")
print(f"\n  BF16 has 8-bit mantissa (vs FP32's 23-bit), so slight rounding may occur.")

PHASE 4: Cast to BF16

Operation: output.to(torch.bfloat16)
  [4, 4] FP32 → [4, 4] BF16

FP32 output =
tensor([[ 0.7245,  0.2939, -0.0134,  0.1990],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.1763,  0.3185, -0.1247,  0.0109]])

BF16 output =
tensor([[ 0.7227,  0.2930, -0.0134,  0.1992],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.1768,  0.3184, -0.1245,  0.0109]], dtype=torch.bfloat16)

  BF16 has 8-bit mantissa (vs FP32's 23-bit), so slight rounding may occur.


---
# COMPLETE SUMMARY

In [15]:
print("="*70)
print("FULL MoE KERNEL PIPELINE SUMMARY")
print("="*70)
print(f"""
PHASE 1: FP8 Block-Scale Dequantization
  For each tensor (A, W13, W2):
    1. Cast FP8 → FP32              (type conversion, no scaling)
    2. Cast scale → FP32            (ensure dtype)
    3. Permute scale [blocks,T]→[T,blocks]  (transpose for token-first layout)
    4. Unsqueeze(-1)                (add dim for repeat, O(1))
    5. Repeat(1,1,BLOCK)            (replicate each scale BLOCK times)
    6. Reshape(T, dim)              (flatten blocks*BLOCK→dim, O(1))
    7. Multiply fp32 * scale_expanded (elementwise dequant)

  Results:
    A   [{T},{H}] FP32
    W13 [{E_local},{2*I},{H}] FP32
    W2  [{E_local},{H},{I}] FP32

PHASE 2: No-Aux Routing
  2.1 s = sigmoid(logits)           [{T},{E_global}]
  2.2 s_with_bias = s + bias        [{T},{E_global}]
  2.3 Group scoring:
      view as [{T},{N_GROUP},{group_size}]
      top-2 per group → sum → group_scores [{T},{N_GROUP}]
  2.4 Select top {TOPK_GROUP} groups:
      topk → group_mask → score_mask [{T},{E_global}]
  2.5 Global top-{TOP_K}:
      masked_fill(-inf) → topk → topk_idx [{T},{TOP_K}]
  2.6 Weights:
      M = scatter(topk_idx)
      weights = normalize(s * M) * scale_factor  [{T},{E_global}]

PHASE 3: Per-Expert Compute Loop (×{E_local} local experts)
  For each local expert le (global expert ge):
    3a. Token selection:
        sel_mask = (topk_idx == ge).any(dim=1)  → [T] bool
        token_idx = nonzero(sel_mask)            → [Tk]
    3b. Gather:
        A_e = A[token_idx]  [{'{'}Tk{'}'},H], W13_e = W13[le], W2_e = W2[le]
    3c. GEMM1:
        G1 = A_e @ W13_e.T  [{'{'}Tk{'}'},H]×[H,2I] → [{'{'}Tk{'}'},2I]
    3d. SwiGLU:
        X1, X2 = split(G1)
        C = silu(X2) * X1   [{'{'}Tk{'}'},I]
    3e. GEMM2:
        O = C @ W2_e.T      [{'{'}Tk{'}'},I]×[I,H] → [{'{'}Tk{'}'},H]
    3f. Weighted accumulate:
        output[token_idx] += O * weights[token_idx, ge]

PHASE 4: Cast
  output = output_fp32.to(bfloat16)  [{T},{H}] BF16
""")

print("FINAL OUTPUT:")
print(f"  Shape: {list(output.shape)}")
print(f"  Dtype: {output.dtype}")
print(f"  Values:\n{output}")

FULL MoE KERNEL PIPELINE SUMMARY

PHASE 1: FP8 Block-Scale Dequantization
  For each tensor (A, W13, W2):
    1. Cast FP8 → FP32              (type conversion, no scaling)
    2. Cast scale → FP32            (ensure dtype)
    3. Permute scale [blocks,T]→[T,blocks]  (transpose for token-first layout)
    4. Unsqueeze(-1)                (add dim for repeat, O(1))
    5. Repeat(1,1,BLOCK)            (replicate each scale BLOCK times)
    6. Reshape(T, dim)              (flatten blocks*BLOCK→dim, O(1))
    7. Multiply fp32 * scale_expanded (elementwise dequant)

  Results:
    A   [4,4] FP32
    W13 [2,4,4] FP32
    W2  [2,4,2] FP32

PHASE 2: No-Aux Routing
  2.1 s = sigmoid(logits)           [4,8]
  2.2 s_with_bias = s + bias        [4,8]
  2.3 Group scoring:
      view as [4,2,4]
      top-2 per group → sum → group_scores [4,2]
  2.4 Select top 1 groups:
      topk → group_mask → score_mask [4,8]
  2.5 Global top-2:
      masked_fill(-inf) → topk → topk_idx [4,2]
  2.6 Weights:
      M 